In [ ]:
import matplotlib.pyplot as plt

INSIDE = 0
LEFT = 1
RIGHT = 2
BOTTOM = 4
TOP = 8

X_MIN, Y_MIN, X_MAX, Y_MAX = 2, 2, 6, 6


def compute_code(x, y):
    code = INSIDE

    if x < X_MIN:
        code |= LEFT
    elif x > X_MAX:
        code |= RIGHT

    if y < Y_MIN:
        code |= BOTTOM
    elif y > Y_MAX:
        code |= TOP

    return code


def cohen_sutherland_clip(x1, y1, x2, y2):
    code1 = compute_code(x1, y1)
    code2 = compute_code(x2, y2)

    accept = False

    while True:
        if code1 == 0 and code2 == 0:
            accept = True
            break

        elif (code1 & code2) != 0:
            break

        else:
            x, y = 0.0, 0.0

            code_out = code1 if code1 != 0 else code2

            if code_out & TOP:
                x = x1 + (x2 - x1) * (Y_MAX - y1) / (y2 - y1)
                y = Y_MAX

            elif code_out & BOTTOM:
                x = x1 + (x2 - x1) * (Y_MIN - y1) / (y2 - y1)
                y = Y_MIN

            elif code_out & RIGHT:
                y = y1 + (y2 - y1) * (X_MAX - x1) / (x2 - x1)
                x = X_MAX

            elif code_out & LEFT:
                y = y1 + (y2 - y1) * (X_MIN - x1) / (x2 - x1)
                x = X_MIN

            if code_out == code1:
                x1, y1 = x, y
                code1 = compute_code(x1, y1)
            else:
                x2, y2 = x, y
                code2 = compute_code(x2, y2)

    if accept:
        return x1, y1, x2, y2
    else:
        return None, None, None, None


def liang_barsky_clip(x1, y1, x2, y2):
    dx = x2 - x1
    dy = y2 - y1

    p = [-dx, dx, -dy, dy]
    q = [x1 - X_MIN, X_MAX - x1, y1 - Y_MIN, Y_MAX - y1]

    t_enter = 0.0
    t_exit = 1.0

    for i in range(4):
        if p[i] == 0:
            if q[i] < 0:
                return None, None, None, None
        else:
            t = q[i] / p[i]

            if p[i] < 0:
                t_enter = max(t_enter, t)
            else:
                t_exit = min(t_exit, t)

    if t_enter > t_exit:
        return None, None, None, None

    nx1 = x1 + t_enter * dx
    ny1 = y1 + t_enter * dy

    nx2 = x1 + t_exit * dx
    ny2 = y1 + t_exit * dy

    return nx1, ny1, nx2, ny2


def plot_lines(original_lines, clipped_lines_cs, clipped_lines_lb, title, ax):
    ax.set_title(title)

    padding = 2

    xs = []
    ys = []

    for line in original_lines:
        xs.extend([line[0], line[2]])
        ys.extend([line[1], line[3]])

    ax.set_xlim(min(X_MIN, min(xs)) - padding,
                max(X_MAX, max(xs)) + padding)

    ax.set_ylim(min(Y_MIN, min(ys)) - padding,
                max(Y_MAX, max(ys)) + padding)

    ax.set_aspect('equal', adjustable='box')

    ax.plot(
        [X_MIN, X_MAX, X_MAX, X_MIN, X_MIN],
        [Y_MIN, Y_MIN, Y_MAX, Y_MAX, Y_MIN],
        'k-',
        linewidth=2,
        label='Clipping Window'
    )

    for line in original_lines:
        ax.plot(
            [line[0], line[2]],
            [line[1], line[3]],
            'b-.',
            alpha=0.5,
            label='Original Line'
        )

    for line in clipped_lines_cs:
        ax.plot(
            [line[0], line[2]],
            [line[1], line[3]],
            'r-',
            linewidth=3,
            label='Cohen-Sutherland'
        )

    for line in clipped_lines_lb:
        ax.plot(
            [line[0], line[2]],
            [line[1], line[3]],
            'g-',
            linewidth=3,
            label='Liang-Barsky'
        )

    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys())

    ax.grid(True)


# Input lines
lines_to_clip = []

num_lines = int(input("Enter the number of lines you want to clip: "))

for i in range(num_lines):
    while True:
        try:
            coords_str = input(
                f"Enter coordinates for line {i + 1} (x1 y1 x2 y2): "
            )

            x1, y1, x2, y2 = map(float, coords_str.split())

            lines_to_clip.append((x1, y1, x2, y2))
            break

        except ValueError:
            print(
                "Invalid input. Please enter four numbers separated by spaces."
            )


clipped_cs_results = []
clipped_lb_results = []

for x1, y1, x2, y2 in lines_to_clip:
    cs_res = cohen_sutherland_clip(x1, y1, x2, y2)

    if cs_res[0] is not None:
        clipped_cs_results.append(cs_res)

    lb_res = liang_barsky_clip(x1, y1, x2, y2)

    if lb_res[0] is not None:
        clipped_lb_results.append(lb_res)


fig, axes = plt.subplots(1, 2, figsize=(15, 7))

plot_lines(
    lines_to_clip,
    clipped_cs_results,
    [],
    "Cohen-Sutherland Line Clipping",
    axes[0]
)

plot_lines(
    lines_to_clip,
    [],
    clipped_lb_results,
    "Liang-Barsky Line Clipping",
    axes[1]
)

plt.tight_layout()
plt.show()


print(
    f"Clipping Window: "
    f"(X_MIN={X_MIN}, Y_MIN={Y_MIN}, "
    f"X_MAX={X_MAX}, Y_MAX={Y_MAX})"
)

print("\nCohen-Sutherland Clipped Lines:")

for i, line in enumerate(lines_to_clip):
    cs_res = cohen_sutherland_clip(*line)

    print(
        f"Original Line {i+1}: "
        f"({line[0]:.2f}, {line[1]:.2f}) - "
        f"({line[2]:.2f}, {line[3]:.2f})"
    )

    if cs_res[0] is not None:
        print(
            f"Clipped: "
            f"({cs_res[0]:.2f}, {cs_res[1]:.2f}) - "
            f"({cs_res[2]:.2f}, {cs_res[3]:.2f})"
        )
    else:
        print("Clipped: Rejected")


print("\nLiang-Barsky Clipped Lines:")

for i, line in enumerate(lines_to_clip):
    lb_res = liang_barsky_clip(*line)

    print(
        f"Original Line {i+1}: "
        f"({line[0]:.2f}, {line[1]:.2f}) - "
        f"({line[2]:.2f}, {line[3]:.2f})"
    )

    if lb_res[0] is not None:
        print(
            f"Clipped: "
            f"({lb_res[0]:.2f}, {lb_res[1]:.2f}) - "
            f"({lb_res[2]:.2f}, {lb_res[3]:.2f})"
        )
    else:
        print("Clipped: Rejected")